In [ ]:
# Cell 1 (Kaggle, T4 x2): setup (git + Tiny-ImageNet via kagglehub + speed patches). Safe to re-run.
import os, sys, shutil, base64, tarfile, subprocess as sp
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import torch, yaml
assert torch.cuda.device_count() >= 1, "Settings > Accelerator > GPU T4 x2"

UP, FORK = "https://github.com/Nikhub7-dotcom/MinorProjectV3.git", "https://github.com/shivamm-verma/MinorProjectV3.git"
BR, REPO, DATA = "shivam/imagenet-results", "/tmp/MinorProjectV3", "/tmp/data"
AUTH = "http.https://github.com/.extraheader=AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{UserSecretsClient().get_secret('GITHUB_TOKEN')}".encode()).decode()

def run(cmd, cwd=None, show=True):
    p = sp.Popen(cmd, cwd=cwd, stdout=sp.PIPE, stderr=sp.STDOUT, text=True, bufsize=1); out = []
    for l in p.stdout:
        out.append(l); show and print(l, end="")
    return p.wait(), "".join(out)

def g(*a, auth=False, show=False):
    return run(["git", *(["-c", AUTH] if auth else []), *a], cwd=PROJ, show=show)

if not (Path(REPO) / ".git").is_dir():
    shutil.rmtree(REPO, ignore_errors=True)
    assert run(["git", "-c", AUTH, "clone", UP, REPO])[0] == 0
PROJ = next(str(p) for p in (Path(REPO), Path(REPO) / "poison-cifar") if (p / "src" / "run_experiment.py").is_file())
for n, u in (("origin", FORK), ("upstream", UP)):
    g("remote", "add", n, u); g("remote", "set-url", n, u)
g("config", "user.name", "Shivam Verma"); g("config", "user.email", "shivam.verma256@outlook.com")

g("reset", "-q", "--hard")
if g("rev-parse", "--verify", "-q", BR)[0] == 0: g("checkout", "-q", BR); base = f"existing local {BR}"
else:
    base = None
    for r in ("upstream", "origin"):
        if g("fetch", r, BR, auth=True)[0] == 0: base = f"{r}/{BR}"; break
    if base is None: g("fetch", "upstream", "main", auth=True); base = "upstream/main"
    assert g("checkout", "-q", "-B", BR, base, show=True)[0] == 0
print("base:", base)
PUSH_ORDER = [r for r in ("upstream", "origin") if g("push", "--dry-run", r, f"HEAD:refs/heads/{BR}", auth=True)[0] == 0]
print("writable remotes (in push order):", PUSH_ORDER)
assert PUSH_ORDER, "Token cannot push to either repo: give GITHUB_TOKEN 'Contents: read/write' (fine-grained) or 'repo' scope (classic)."

# Imagenette (for imagenet_subset)
if not (Path(DATA, "train").is_dir() and Path(DATA, "val").is_dir()):
    os.makedirs(DATA, exist_ok=True)
    run(["wget", "-q", "-O", "/tmp/im.tgz", "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"])
    tarfile.open("/tmp/im.tgz").extractall(DATA, filter="data")
    for s in ("train", "val"): shutil.move(f"{DATA}/imagenette2-160/{s}", f"{DATA}/{s}")

# Tiny-ImageNet from Kaggle -> the repo expects <data_root>/tiny-imagenet-200/train/<class>/...
run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
import kagglehub
KP = kagglehub.dataset_download("akash2sharma/tiny-imagenet"); print("kaggle path:", KP)
tr = next(Path(r) for r, d, _ in os.walk(KP) if Path(r).name == "train" and len(d) >= 200)
os.makedirs(DATA, exist_ok=True); link = Path(DATA, "tiny-imagenet-200")
link.unlink(missing_ok=True); link.symlink_to(tr.parent, target_is_directory=True)
print("train dir:", tr, "| classes:", len(os.listdir(tr)))

# Patched COPY of src (smaller eval batch -> no OOM). Repo files stay untouched, so git stays clean.
shutil.rmtree("/tmp/src_run", ignore_errors=True); shutil.copytree(f"{PROJ}/src", "/tmp/src_run")
run(["sed", "-i", "s/batch_size=256/batch_size=128/g", "/tmp/src_run/run_experiment.py"])
# Speed wrapper: cudnn.benchmark + channels_last, and clean/ASR eval only every EVAL_EVERY epochs (always on the last one).
open("/tmp/src_run/fast_run.py", "w").write('''import os, sys, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import run_experiment as rx
torch.backends.cudnn.benchmark = True
K, E, ck = int(os.environ["EVAL_EVERY"]), int(os.environ["EPOCHS"]), os.environ["CKPT"]
st = {"e": torch.load(ck, map_location="cpu", weights_only=False)["epoch"] if os.path.exists(ck) else 0}
_t, _c, _a, _b = rx.train_one_epoch, rx.evaluate_clean, rx.evaluate_asr, rx.build_model
def train(*a, **k): st["e"] += 1; return _t(*a, **k)
skip = lambda: st["e"] % K != 0 and st["e"] != E
rx.train_one_epoch = train
rx.evaluate_clean = lambda *a, **k: (float("nan"), float("nan")) if skip() else _c(*a, **k)
rx.evaluate_asr = lambda *a, **k: float("nan") if skip() else _a(*a, **k)
rx.build_model = lambda *a, **k: _b(*a, **k).to(memory_format=torch.channels_last)
rx.main()
''')
shutil.rmtree(f"{PROJ}/checkpoints", ignore_errors=True); shutil.rmtree(f"{PROJ}/logs", ignore_errors=True)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "| SETUP OK")

In [ ]:
# Cell 2 (Kaggle T4 x2): resnet50 backdoor, imagenet_subset + tinyimagenet, rates 0.5% -> 1% -> 0.1%.
# One job per GPU, results pushed as soon as each finishes. Finished ones (already on the branch) are skipped. Re-run to resume.
import time, yaml, glob, threading, queue
from pathlib import Path
MODEL, EVAL_EVERY = "resnet50", 10
JOBS = [(d, r) for d in ("imagenet_subset", "tinyimagenet") for r in ("0_5pct", "1pct", "0_1pct")]
NGPU = max(1, torch.cuda.device_count()); BS, status, lock, q, t_all = {}, {}, threading.Lock(), queue.Queue(), time.time()

def push(run_id):
    with lock:
        g("add", "-f", f"results/{run_id}_result.json"); g("commit", "-q", "-m", f"Add {run_id} result")
        for r in PUSH_ORDER:
            for _ in range(3):
                if g("push", "-u", r, BR, auth=True, show=True)[0] == 0: return r
                g("pull", "--rebase", "--autostash", r, BR, auth=True); time.sleep(10)

def train(cfg, run_id, epochs, key, gpu):
    for attempt in range(6):
        bs = BS.setdefault(key, 128); lr = 0.1 * bs / 128       # linear lr scaling if batch had to shrink
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "EVAL_EVERY": str(EVAL_EVERY), "EPOCHS": str(epochs), "CKPT": f"{PROJ}/checkpoints/{run_id}.pt"}
        p = sp.Popen([sys.executable, "-u", "/tmp/src_run/fast_run.py", "--config", cfg, "--data-root", DATA,
                      "--num-workers", str(max(1, os.cpu_count() // NGPU)), "--batch-size", str(bs), "--lr", str(lr)],
                     cwd=PROJ, env=env, stdout=sp.PIPE, stderr=sp.STDOUT, text=True, bufsize=1)
        oom = False
        for l in p.stdout:
            print(f"[gpu{gpu} {run_id.replace('_backdoor_', ' ')}] {l}", end=""); oom |= "out of memory" in l.lower()
        if p.wait() == 0: return True
        if oom and bs > 8: BS[key] = min(BS[key], bs // 2); print(f"\n>>> [{run_id}] OOM at batch {bs}; retrying with {BS[key]}\n")
        else: print(f"\n>>> [{run_id}] failed (attempt {attempt + 1}); retrying (resumes from checkpoint)\n"); time.sleep(15)
    return False

def worker(gpu):
    while True:
        try: ds, rate = q.get_nowait()
        except queue.Empty: return
        cfg = f"configs/experiments/{ds}_{MODEL}_backdoor_{rate}.yaml"; c = yaml.safe_load(open(f"{PROJ}/{cfg}"))
        run_id, t0 = f"{c['dataset']}_{c['model']}_{c['attack']}_{str(c['rate']).replace('.', 'p')}", time.time()
        res = Path(PROJ, "results", f"{run_id}_result.json")
        print(f"\n>>> GPU{gpu} starting {run_id}" + (" (result exists, skipping training)" if res.is_file() else ""))
        if not res.is_file() and not train(cfg, run_id, c["epochs"], ds, gpu): status[run_id] = "TRAIN FAILED"; continue
        if not res.is_file(): status[run_id] = "NO RESULT FILE"; continue
        dest = push(run_id)
        status[run_id] = f"pushed -> {dest}" if dest else "TRAINED BUT PUSH FAILED (result still on this machine)"
        for f in glob.glob(f"{PROJ}/checkpoints/{run_id}*"): os.remove(f)
        print(f">>> {run_id}: {status[run_id]} ({(time.time() - t0) / 60:.1f} min)")

for j in JOBS: q.put(j)
ts = [threading.Thread(target=worker, args=(i,)) for i in range(NGPU)]
[t.start() for t in ts]; [t.join() for t in ts]
print("\n" + "=" * 70 + f"\nFINISHED in {(time.time() - t_all) / 3600:.2f} h on branch {BR}")
for k, v in status.items(): print(f"  {k}: {v}")

In [ ]:
# Cell 2 (Kaggle T4 x2): MNIST resnet10 backdoor, rates 0.5% -> 1% -> 0.1%.

# complete this by tomorrow afternoon
# [IMPORTANT TASK to be done]

import time, yaml, glob, threading, queue
from pathlib import Path
EVAL_EVERY = 10
JOBS = [("mnist", "resnet10", r) for r in ("0_5pct", "1pct", "0_1pct")]
NGPU = max(1, torch.cuda.device_count()); BS, status, lock, q, t_all = {}, {}, threading.Lock(), queue.Queue(), time.time()

def push(run_id):
    with lock:
        g("add", "-f", f"results/{run_id}_result.json"); g("commit", "-q", "-m", f"Add {run_id} result")
        for r in PUSH_ORDER:
            for _ in range(3):
                if g("push", "-u", r, BR, auth=True, show=True)[0] == 0: return r
                g("pull", "--rebase", "--autostash", r, BR, auth=True); time.sleep(10)

def train(cfg, run_id, epochs, key, gpu):
    for attempt in range(6):
        bs = BS.setdefault(key, 128); lr = 0.1 * bs / 128       # linear lr scaling if batch had to shrink
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "EVAL_EVERY": str(EVAL_EVERY), "EPOCHS": str(epochs), "CKPT": f"{PROJ}/checkpoints/{run_id}.pt"}
        p = sp.Popen([sys.executable, "-u", "/tmp/src_run/fast_run.py", "--config", cfg, "--data-root", DATA,
                      "--num-workers", str(max(1, os.cpu_count() // NGPU)), "--batch-size", str(bs), "--lr", str(lr)],
                     cwd=PROJ, env=env, stdout=sp.PIPE, stderr=sp.STDOUT, text=True, bufsize=1)
        oom = False
        for l in p.stdout:
            print(f"[gpu{gpu} {run_id.replace('_backdoor_', ' ')}] {l}", end=""); oom |= "out of memory" in l.lower()
        if p.wait() == 0: return True
        if oom and bs > 8: BS[key] = min(BS[key], bs // 2); print(f"\n>>> [{run_id}] OOM at batch {bs}; retrying with {BS[key]}\n")
        else: print(f"\n>>> [{run_id}] failed (attempt {attempt + 1}); retrying (resumes from checkpoint)\n"); time.sleep(15)
    return False

def worker(gpu):
    while True:
        try: ds, model, rate = q.get_nowait()
        except queue.Empty: return
        cfg = f"configs/experiments/{ds}_{model}_backdoor_{rate}.yaml"; c = yaml.safe_load(open(f"{PROJ}/{cfg}"))
        run_id, t0 = f"{c['dataset']}_{c['model']}_{c['attack']}_{str(c['rate']).replace('.', 'p')}", time.time()
        res = Path(PROJ, "results", f"{run_id}_result.json")
        print(f"\n>>> GPU{gpu} starting {run_id}" + (" (result exists, skipping training)" if res.is_file() else ""))
        if not res.is_file() and not train(cfg, run_id, c["epochs"], ds, gpu): status[run_id] = "TRAIN FAILED"; continue
        if not res.is_file(): status[run_id] = "NO RESULT FILE"; continue
        dest = push(run_id)
        status[run_id] = f"pushed -> {dest}" if dest else "TRAINED BUT PUSH FAILED (result still on this machine)"
        for f in glob.glob(f"{PROJ}/checkpoints/{run_id}*"): os.remove(f)
        print(f">>> {run_id}: {status[run_id]} ({(time.time() - t0) / 60:.1f} min)")

for j in JOBS: q.put(j)
ts = [threading.Thread(target=worker, args=(i,)) for i in range(NGPU)]
[t.start() for t in ts]; [t.join() for t in ts]
print("\n" + "=" * 70 + f"\nFINISHED in {(time.time() - t_all) / 3600:.2f} h on branch {BR}")
for k, v in status.items(): print(f"  {k}: {v}")